# Task 2: Predictive Analysis with Machine Learning

This notebook builds a regression model to predict supermarket sales and demonstrates feature selection.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_path = 'c:/Users/Admin/OneDrive/Documents/BigData project/SuperMarket Analysis.csv'
pandas_df = pd.read_csv(data_path)

model_df = pandas_df.copy()
model_df['Date'] = pd.to_datetime(model_df['Date'])
model_df['Month'] = model_df['Date'].dt.month

feature_cols = [
    'Unit price', 'Quantity', 'Tax 5%', 'Month',
    'Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment'
]
target_col = 'Sales'

X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_cols = ['Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment']
numeric_cols = ['Unit price', 'Quantity', 'Tax 5%', 'Month']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

all_features = preprocessor.fit_transform(X_train)
test_features = preprocessor.transform(X_test)

selector = SelectKBest(score_func=f_regression, k=10)
selected_train = selector.fit_transform(all_features, y_train)
selected_test = selector.transform(test_features)

lr_model = LinearRegression()
lr_model.fit(selected_train, y_train)
predictions = lr_model.predict(selected_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

selected_feature_names = np.array([f'feature_{i}' for i in range(all_features.shape[1])])[selector.get_support()]

print('Selected feature count:', len(selected_feature_names))
print('Top selected feature indices:', np.where(selector.get_support())[0][:10].tolist())
print('RMSE:', round(rmse, 3))
print('MAE:', round(mae, 3))
print('R2:', round(r2, 3))

## Interpretation

- The model predicts sales from pricing, quantity, tax, month, and customer/product metadata.
- `SelectKBest` keeps the most informative transformed features.
- Lower RMSE and higher R² indicate a better fit.